# 02 — Load Cleaned Data into Postgres
connect → create tables → load data → verify counts.


## Cell 1 — Imports & connection

In [4]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()  # reads the .env file sitting in your project root

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    print("Connected:", conn.execute(text("SELECT version();")).fetchone()[0][:30])


Connected: PostgreSQL 18.4 on x86_64-wind


## Cell 2 — Load the cleaned CSVs from Notebook 01

In [5]:
orders = pd.read_csv("../outputs/orders_clean.csv", parse_dates=["Order Date", "Ship Date"])
returns = pd.read_csv("../outputs/returns_clean.csv")
people = pd.read_csv("../outputs/people_clean.csv")

print(orders.shape, returns.shape, people.shape)


(51290, 26) (1079, 3) (23, 2)


## Cell 3 — Create the tables (runs sql/01_schema.sql)

In [6]:
with open("../sql/01_schema.sql") as f:
    schema_sql = f.read()

with engine.connect() as conn:
    conn.execute(text(schema_sql))
    conn.commit()

print("Tables created.")


Tables created.


## Cell 4 — Shape the data into each table

In [ ]:
dim_customer = orders[["Customer ID", "Customer Name", "Segment"]].drop_duplicates(subset="Customer ID")
dim_customer.columns = ["customer_id", "customer_name", "segment"]

dim_product = orders[["Product ID", "Product Name", "Category", "Sub-Category"]].drop_duplicates(subset="Product ID")
dim_product.columns = ["product_id", "product_name", "category", "sub_category"]

geo_cols = ["City", "State", "Country", "Region", "Market", "Postal Code"]
dim_geography = orders[geo_cols].drop_duplicates().reset_index(drop=True)
dim_geography.insert(0, "geography_key", dim_geography.index + 1)
dim_geography.columns = ["geography_key", "city", "state", "country", "region", "market", "postal_code"]

dim_manager = people.copy()
dim_manager.columns = ["region", "manager_name"]
assert dim_manager["region"].is_unique, "dim_manager region is not unique — Canada collapse in Notebook 01 may not have run."

bridge_returns = returns.copy()
bridge_returns.columns = ["returned", "order_id", "region"]

# attach geography_key to each order line before building the fact table
orders_geo = orders.merge(
    dim_geography, left_on=geo_cols,
    right_on=["city", "state", "country", "region", "market", "postal_code"],
    how="left"
)


assert len(orders_geo) == len(orders), (
    f"Geography merge changed row count: {len(orders):,} -> {len(orders_geo):,}. "
    "This means the merge matched more than one geography row for at least one order line."
)
assert orders_geo["geography_key"].notna().all(), "Some order lines did not match any geography_key."
print(f"Geography merge OK: {len(orders_geo):,} rows in, {len(orders_geo):,} rows out, 0 unmatched.")

fact_order_lines = pd.DataFrame({
    "row_id": orders_geo["Row ID"],
    "order_id": orders_geo["Order ID"],
    "order_date": orders_geo["Order Date"],
    "ship_date": orders_geo["Ship Date"],
    "ship_mode": orders_geo["Ship Mode"],
    "customer_id": orders_geo["Customer ID"],
    "product_id": orders_geo["Product ID"],
    "geography_key": orders_geo["geography_key"],
    "order_priority": orders_geo["Order Priority"],
    "sales": orders_geo["Sales"],
    "quantity": orders_geo["Quantity"],
    "discount": orders_geo["Discount"],
    "profit": orders_geo["Profit"],
    "shipping_cost": orders_geo["Shipping Cost"],
    "delivery_days": orders_geo["Delivery Days"],
    "is_returned_line": orders_geo["Is Returned Line"].astype(int),
})


assert len(fact_order_lines) == len(orders), "fact_order_lines row count does not match source Orders."
assert fact_order_lines["row_id"].is_unique, "row_id is not unique in fact_order_lines — grain is broken."
assert fact_order_lines["row_id"].notna().all(), "Some fact rows have a null row_id."
print(f"Fact table grain OK: {len(fact_order_lines):,} rows, row_id unique, no nulls.")

print("\nRows ready to load:")
print("dim_customer:", len(dim_customer))
print("dim_product:", len(dim_product))
print("dim_geography:", len(dim_geography))
print("dim_manager:", len(dim_manager))
print("bridge_returns:", len(bridge_returns))
print("fact_order_lines:", len(fact_order_lines))


Geography merge OK: 51,290 rows in, 51,290 rows out, 0 unmatched.
Fact table grain OK: 51,290 rows, row_id unique, no nulls.

Rows ready to load:
dim_customer: 17415
dim_product: 3788
dim_geography: 3856
dim_manager: 23
bridge_returns: 1079
fact_order_lines: 51290


## Cell 5 — Load everything into Postgres

In [8]:
dim_customer.to_sql("dim_customer", engine, if_exists="append", index=False)
dim_product.to_sql("dim_product", engine, if_exists="append", index=False)
dim_geography.to_sql("dim_geography", engine, if_exists="append", index=False)
dim_manager.to_sql("dim_manager", engine, if_exists="append", index=False)
bridge_returns.to_sql("bridge_returns", engine, if_exists="append", index=False)
fact_order_lines.to_sql("fact_order_lines", engine, if_exists="append", index=False)

print("All tables loaded.")


All tables loaded.


## Cell 6 — Verify (row counts must match what we loaded)

In [ ]:
with engine.connect() as conn:
    for tbl in ["dim_customer", "dim_product", "dim_geography", "dim_manager", "bridge_returns", "fact_order_lines"]:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
        print(f"{tbl}: {n:,} rows in Postgres")

with engine.connect() as conn:
    orphans = conn.execute(text(
        "SELECT COUNT(*) FROM fact_order_lines WHERE geography_key IS NULL"
    )).scalar()
print(f"\nFact rows with unmapped geography: {orphans} (should be 0)")
assert orphans == 0


dim_customer: 17,415 rows in Postgres
dim_product: 3,788 rows in Postgres
dim_geography: 3,856 rows in Postgres
dim_manager: 23 rows in Postgres
bridge_returns: 1,079 rows in Postgres
fact_order_lines: 51,290 rows in Postgres

Fact rows with unmapped geography: 0 (should be 0)
